# RAG-based Document Q&A (CSV-first)

This notebook demonstrates a **hybrid strategy**:

- **RAG path (Upstash Vector + LLM)** for semantic/document-style questions
- **Exact analytics path (pandas)** for numeric aggregation questions (max/min/avg/sum/count)

Run cells top to bottom for the cleanest experience.

In [ ]:
# Step 1: Load environment variables (.env) for API keys and Upstash credentials.
from pathlib import Path
from dotenv import load_dotenv
import os


def load_env_upwards(filename=".env", max_up=5):
    """Search current folder and parent folders for .env, then load it."""
    p = Path.cwd()
    for _ in range(max_up + 1):
        candidate = p / filename
        if candidate.exists():
            # override=True lets notebook runs pick up the latest .env edits.
            load_dotenv(candidate, override=True)
            return candidate
        p = p.parent
    return None


# Execute the loader and print diagnostics so failures are obvious early.
env_path = load_env_upwards()
print("cwd:", Path.cwd())
print("loaded .env from:", env_path)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("UPSTASH_VECTOR_REST_URL set:", bool(os.getenv("UPSTASH_VECTOR_REST_URL")))
print("UPSTASH_VECTOR_REST_TOKEN set:", bool(os.getenv("UPSTASH_VECTOR_REST_TOKEN")))

cwd: c:\Users\1036506\OneDrive - Blue Yonder\Desktop\learning_langchain\ch01
loaded: c:\Users\1036506\OneDrive - Blue Yonder\Desktop\learning_langchain\ch01\.env
UPSTASH_VECTOR_REST_URL set: True
UPSTASH_VECTOR_REST_TOKEN set: True


In [6]:
# Step 2: Initialize embeddings + vector store connection.
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.upstash import UpstashVectorStore

# text-embedding-3-small produces 1536-dim vectors.
# This must match the Upstash index dimension you created.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# UpstashVectorStore reads UPSTASH_VECTOR_REST_URL/TOKEN from environment.
store = UpstashVectorStore(embedding=embeddings)
print("store ready")

store ready


In [7]:
# Step 3: Smoke test the index with tiny sample texts.
texts = [
    "LangChain is a framework for building LLM applications.",
    "FastAPI is a web framework for building APIs with Python.",
]

# Metadata is stored alongside vectors and helps with traceability/citations.
metadatas = [{"source": "smoke_test", "chunk": i} for i in range(len(texts))]

# add_texts computes embeddings and upserts vectors to Upstash.
ids = store.add_texts(texts, metadatas=metadatas)
print("Inserted ids:", ids)

# similarity_search returns the most semantically relevant text chunks.
results = store.similarity_search("What is FastAPI?", k=2)
for doc in results:
    print("\n---")
    print("content:", doc.page_content)
    print("metadata:", doc.metadata)

Inserted ids: ['31acc8f3-cfaf-481f-814c-a40a8c89561f', '63aeca7e-5c05-4418-afe6-9201b5f00d3e']


In [8]:
#check retrieval

results = store.similarity_search("What is FastAPI?", k=2)
for doc in results:
    print("\n---")
    print("content:", doc.page_content)
    print("metadata:", doc.metadata)


---
content: FastAPI is a web framework for building APIs with Python.
metadata: {'source': 'smoke_test', 'chunk': 1}

---
content: LangChain is a framework for building LLM applications.
metadata: {'source': 'smoke_test', 'chunk': 0}


## CSV Ingestion Pipeline

This section prepares a large CSV for vector indexing in a memory-safe way.

1. Preview schema (without loading full file)
2. Estimate row count
3. Convert rows to buffered text chunks (`Document` objects)
4. Upsert to Upstash in batches

In [10]:
# Step 4: Load CSV metadata safely (without loading 194MB fully into memory).
# Run once per kernel to ensure pandas is available.
%pip install -q pandas

import pandas as pd
from pathlib import Path
from langchain_core.documents import Document

# Use raw string (r"...") for Windows paths to avoid backslash escape errors.
csv_path = Path(r"C:\Users\1036506\Downloads\data_M5\sell_prices.csv")

# Read only a few rows for preview/column inspection.
preview_df = pd.read_csv(csv_path, nrows=5)
print("preview rows:", len(preview_df), "| cols:", list(preview_df.columns))
preview_df.head(3)

Note: you may need to restart the kernel to use updated packages.
rows: 6841121 | cols: ['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26


In [12]:
# Step 5: Estimate row count without loading all columns into memory.
# This scans the file once and subtracts 1 header row.
with open(csv_path, "r", encoding="utf-8") as f:
    row_count = sum(1 for _ in f) - 1

print("row_count:", row_count)

6841121

In [13]:
# Step 6: Convert a huge CSV into LangChain Documents in a memory-safe way.
# Strategy: read CSV in chunks, then pack many rows into one text block per vector.

from langchain_core.documents import Document

# Tune these knobs for speed/cost/quality tradeoff.
max_source_rows = 100_000   # cap for MVP; raise gradually later
chunk_read_size = 5_000     # rows read from disk at a time
target_chars = 4_000        # target text size per vectorized document
max_rows_per_doc = 100      # row limit per vectorized document

docs = []
processed_rows = 0
doc_index = 0

# Buffer accumulates row strings until limits are reached, then flushes to a Document.
buffer = []
buffer_rows = 0
buffer_chars = 0

for chunk in pd.read_csv(csv_path, chunksize=chunk_read_size):
    # Replace NaN to keep text clean and avoid embedding odd tokens.
    chunk = chunk.fillna("")

    for _, row in chunk.iterrows():
        if processed_rows >= max_source_rows:
            break

        # Normalize each row into plain text: col: value | col: value ...
        row_text = " | ".join([f"{col}: {row[col]}" for col in chunk.columns])

        # Flush when either char budget or row budget is reached.
        should_flush = (
            (buffer_chars + len(row_text) > target_chars) or
            (buffer_rows >= max_rows_per_doc)
        )

        if should_flush and buffer:
            docs.append(
                Document(
                    page_content="\n".join(buffer),
                    metadata={
                        "source": str(csv_path),
                        "file_type": "csv",
                        "chunk_index": doc_index,
                        "rows_in_chunk": buffer_rows,
                        "start_row_estimate": processed_rows - buffer_rows,
                    },
                )
            )
            doc_index += 1
            buffer = []
            buffer_rows = 0
            buffer_chars = 0

        buffer.append(row_text)
        buffer_rows += 1
        buffer_chars += len(row_text)
        processed_rows += 1

    if processed_rows >= max_source_rows:
        break

# Flush final partial buffer.
if buffer:
    docs.append(
        Document(
            page_content="\n".join(buffer),
            metadata={
                "source": str(csv_path),
                "file_type": "csv",
                "chunk_index": doc_index,
                "rows_in_chunk": buffer_rows,
                "start_row_estimate": processed_rows - buffer_rows,
            },
        )
    )

print("processed_rows:", processed_rows)
print("documents created:", len(docs))
print("sample doc preview:", docs[0].page_content[:300] if docs else "None")
print("sample metadata:", docs[0].metadata if docs else "None")

processed_rows: 100000
documents created: 1927
sample doc preview: store_id: CA_1 | item_id: HOBBIES_1_001 | wm_yr_wk: 11325 | sell_price: 9.58
store_id: CA_1 | item_id: HOBBIES_1_001 | wm_yr_wk: 11326 | sell_price: 9.58
store_id: CA_1 | item_id: HOBBIES_1_001 | wm_yr_wk: 11327 | sell_price: 8.26
store_id: CA_1 | item_id: HOBBIES_1_001 | wm_yr_wk: 11328 | sell_pric
sample metadata: {'source': 'C:\\Users\\1036506\\Downloads\\data_M5\\sell_prices.csv', 'file_type': 'csv', 'chunk_index': 0, 'rows_in_chunk': 52, 'start_row_estimate': 0}


In [ ]:
# Optional step (usually SKIP for this workflow):
# docs are already chunked in the previous cell.
# Running another splitter here typically creates too many vectors.
# Keep this only if your per-doc text is still very large.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
chunked_docs = splitter.split_documents(docs)
print("chunked docs:", len(chunked_docs), "(optional path)")

In [15]:
# Step 7: Upsert embeddings to Upstash in batches (progress-friendly).
# For large uploads, batching is safer than one giant add_documents call.

docs_to_insert = docs  # use `chunked_docs` only if you intentionally ran optional splitter
batch_size = 200
inserted_ids = []
total = len(docs_to_insert)

for start in range(0, total, batch_size):
    end = min(start + batch_size, total)
    batch = docs_to_insert[start:end]
    batch_ids = store.add_documents(batch)
    inserted_ids.extend(batch_ids)
    print(f"Inserted {end}/{total} ({(end/total)*100:.1f}%)")

print("inserted:", len(inserted_ids))

inserted: 1927


### Expected Output (Step 7)

- Progress logs like `Inserted 400/2200 (18.2%)`
- Final line `inserted: <count>`

If this is slow, reduce `max_source_rows` in Step 6 and rerun ingestion.

In [16]:
# Step 8: Retrieval-only debug.
# This helps inspect what vectors are returned before asking the LLM to answer.
query = "which item has the highest price?"
hits = store.similarity_search_with_score(query, k=4)

for doc, score in hits:
    print("\n---")
    print("score:", round(score, 4))
    print("metadata:", doc.metadata)
    print("content:", doc.page_content[:400])


---
score: 0.6644
metadata: {'source': 'C:\\Users\\1036506\\Downloads\\data_M5\\sell_prices.csv', 'file_type': 'csv', 'chunk_index': 1348, 'rows_in_chunk': 52, 'start_row_estimate': 69953}
content: store_id: CA_1 | item_id: HOBBIES_1_310 | wm_yr_wk: 11523 | sell_price: 4.86
store_id: CA_1 | item_id: HOBBIES_1_310 | wm_yr_wk: 11524 | sell_price: 4.86
store_id: CA_1 | item_id: HOBBIES_1_310 | wm_yr_wk: 11525 | sell_price: 4.86
store_id: CA_1 | item_id: HOBBIES_1_310 | wm_yr_wk: 11526 | sell_price: 4.86
store_id: CA_1 | item_id: HOBBIES_1_310 | wm_yr_wk: 11527 | sell_price: 4.86
store_id: CA_1 

---
score: 0.6636
metadata: {'source': 'C:\\Users\\1036506\\Downloads\\data_M5\\sell_prices.csv', 'file_type': 'csv', 'chunk_index': 1047, 'rows_in_chunk': 51, 'start_row_estimate': 54328}
content: store_id: CA_1 | item_id: HOBBIES_1_243 | wm_yr_wk: 11524 | sell_price: 13.88
store_id: CA_1 | item_id: HOBBIES_1_243 | wm_yr_wk: 11525 | sell_price: 13.88
store_id: CA_1 | item_id: HOBBIES_1_243 | wm_

In [17]:
# Step 9: Basic RAG answer (LLM over retrieved context).
from langchain_openai import ChatOpenAI

# Reuse one model instance; temperature=0 improves consistency for factual tasks.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Build a context block from top retrieval hits.
context = "\n\n".join(
    [f"[{i+1}] {doc.page_content}" for i, (doc, _) in enumerate(hits)]
)

# Constrain the model to answer from context only.
prompt = f"""
You are a data assistant. Answer only using the provided context.
If the answer is not in context, say: "I don't know based on the provided documents."

Context:
{context}

Question:
{query}
"""

answer = llm.invoke(prompt)
print(answer.content)

The item with the highest price is HOBBIES_1_243 with a sell_price of 13.88.


In [18]:
# Step 10: Retrieval audit for a numeric question.
# Useful to prove whether vector hits truly contain evidence for the claim.
query = "Which item has the highest sell_price? Return the item id and sell_price."
hits = store.similarity_search_with_score(query, k=4)

for i, (doc, score) in enumerate(hits, 1):
    print(f"\n--- hit {i} | score={score:.4f} ---")
    print("metadata:", doc.metadata)
    print("content preview:", doc.page_content[:500])


--- hit 1 | score=0.6787 ---
metadata: {'source': 'C:\\Users\\1036506\\Downloads\\data_M5\\sell_prices.csv', 'file_type': 'csv', 'chunk_index': 1803, 'rows_in_chunk': 52, 'start_row_estimate': 93570}
content preview: store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11524 | sell_price: 2.08
store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11525 | sell_price: 2.08
store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11526 | sell_price: 1.98
store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11527 | sell_price: 1.98
store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11528 | sell_price: 1.98
store_id: CA_1 | item_id: HOBBIES_1_412 | wm_yr_wk: 11529 | sell_price: 1.98
store_id: CA_1 | item_id: HOBBIES_1_41

--- hit 2 | score=0.6786 ---
metadata: {'source': 'C:\\Users\\1036506\\Downloads\\data_M5\\sell_prices.csv', 'file_type': 'csv', 'chunk_index': 165, 'rows_in_chunk': 52, 'start_row_estimate': 8570}
content preview: store_id: CA_1 | item_id: HOBBIES_1_036 | wm_yr_wk: 11404 | sell_

In [19]:
# Step 11: Exact analytics path (ground truth) for numeric questions.
# Vector search is approximate semantic retrieval; this computes exact max.
import pandas as pd

use_cols = ["item_id", "sell_price", "store_id", "wm_yr_wk"]
tmp = pd.read_csv(csv_path, usecols=use_cols)

max_price = tmp["sell_price"].max()
rows = tmp[tmp["sell_price"] == max_price].sort_values(["item_id", "store_id", "wm_yr_wk"])

print("Global max sell_price:", max_price)
print("Rows with that max (first 10):")
print(rows.head(10).to_string(index=False))
print("Total rows with max:", len(rows))

Global max sell_price: 107.32
Rows with that max (first 10):
store_id         item_id  wm_yr_wk  sell_price
    WI_3 HOUSEHOLD_2_406     11317      107.32
    WI_3 HOUSEHOLD_2_406     11318      107.32
    WI_3 HOUSEHOLD_2_406     11319      107.32
Total rows with max: 3


### Expected Output (Step 11)

- `Global max sell_price: ...`
- A small table of rows that share that max price
- `Total rows with max: ...`

Use this result as **ground truth** for numeric questions. If RAG gives a different answer, trust this exact analytics result.

## Hybrid Q&A Router (Exact Analytics + RAG)

Use exact tabular computation for aggregation questions (max/min/avg/sum/count/top/group by), and RAG retrieval for semantic/document questions.

In [ ]:
# Deprecated duplicate cell.
# Use the fully annotated hybrid-router cell below (later in this notebook).
print("Skip this cell and run the annotated hybrid router cell below.")

In [ ]:
# Example usage
# Run this after the hybrid-router function cell defines `ask`.
ask_fn = globals().get("ask")
if callable(ask_fn):
    print(ask_fn("What is the highest sell price in the dataset?", str(csv_path), store, llm))
    print("\n" + "=" * 80 + "\n")
    print(ask_fn("Which item_ids around HOBBIES show stable prices in indexed chunks?", str(csv_path), store, llm))
else:
    print("Run the hybrid-router definition cell first (it defines ask()).")

## Fully Annotated Hybrid Router (Learning Version)

This cell mirrors the hybrid strategy with very detailed comments for learning.
You can run this cell as-is; it will redefine the same helper functions.

In [ ]:
# Import Python's regular expression module.
# We use this to detect analytics-like words in user questions.
import re

# Import pandas for exact tabular computations (max/min/avg/group-by/etc.).
import pandas as pd

# Import the chat model wrapper from LangChain.
from langchain_openai import ChatOpenAI

# Create one LLM instance and reuse it across calls.
# temperature=0 makes outputs more deterministic/consistent.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Define regex patterns that indicate an analytics question.
# If a question includes one of these terms, we route it to pandas logic.
ANALYTICS_PATTERNS = [
    r"\bmax\b",        # exact word: max
    r"\bmaximum\b",    # exact word: maximum
    r"\bmin\b",        # exact word: min
    r"\bminimum\b",    # exact word: minimum
    r"\bavg\b",        # avg shorthand
    r"\baverage\b",    # average
    r"\bmean\b",       # mean
    r"\bsum\b",        # sum
    r"\bcount\b",      # count
    r"\btop\b",        # top-N type query
    r"\blowest\b",     # lowest value
    r"\bhighest\b",    # highest value
    r"\bgroup by\b",   # group by aggregation
    r"\bhow many\b",   # count intent in natural language
]


def is_analytics_query(question: str) -> bool:
    """Return True if question likely requires exact table math."""
    # Normalize to lowercase so matching is case-insensitive.
    q = question.lower()

    # Check each regex pattern against the question.
    # any(...) returns True as soon as one pattern matches.
    return any(re.search(pattern, q) for pattern in ANALYTICS_PATTERNS)


def answer_analytics(question: str, csv_path: str) -> str:
    """Answer analytics questions with exact pandas computations."""
    # Lowercase once for easier keyword checks.
    q = question.lower()

    # Handle 'highest/max/maximum' explicitly.
    if "highest" in q or "max" in q or "maximum" in q:
        # Read only needed columns to reduce memory and speed up load.
        df = pd.read_csv(csv_path, usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"])

        # Compute exact global maximum sell_price.
        max_price = df["sell_price"].max()

        # Filter rows where sell_price equals that exact max.
        rows = df[df["sell_price"] == max_price]

        # Sort output for stable, readable presentation.
        rows = rows.sort_values(["item_id", "store_id", "wm_yr_wk"])

        # Build a preview string (first 10 rows) for notebook display.
        preview = rows.head(10).to_string(index=False)

        # Return a formatted answer block.
        return (
            f"Global max sell_price is {max_price}.\n"
            f"Rows with that max (first 10):\n{preview}\n"
            f"Total rows with max: {len(rows)}"
        )

    # Fallback response if analytics intent is detected but not implemented yet.
    return (
        "Analytics query detected, but no exact handler is implemented yet for this question. "
        "Add a specific pandas handler for this operation (for example: average, count, or group by)."
    )


def answer_rag(question: str, store, llm: ChatOpenAI, k: int = 4) -> str:
    """Answer semantic/document questions using vector retrieval + LLM."""
    # Retrieve top-k relevant chunks from Upstash Vector.
    hits = store.similarity_search_with_score(question, k=k)

    # Build context text by concatenating retrieved chunks.
    # We add [1], [2], ... labels for readability.
    context = "\n\n".join(
        [f"[{i+1}] {doc.page_content}" for i, (doc, _) in enumerate(hits)]
    )

    # Keep metadata so we can show traceable sources/citations.
    sources = [doc.metadata for doc, _ in hits]

    # Construct a strict prompt to minimize hallucinations.
    prompt = f"""
You are a document assistant.
Answer ONLY from the provided context.
If the answer is not in context, say: "I don't know based on the indexed documents."

Context:
{context}

Question:
{question}
"""

    # Call the LLM and extract plain text output.
    answer_text = llm.invoke(prompt).content

    # Append source metadata for transparency.
    return answer_text + "\n\nSources:\n" + "\n".join(str(source) for source in sources)


def ask(question: str, csv_path: str, store, llm: ChatOpenAI) -> str:
    """Single entry point that routes to analytics or RAG."""
    # Route analytics-like questions to exact pandas path.
    if is_analytics_query(question):
        return "[route=analytics]\n" + answer_analytics(question, csv_path)

    # Otherwise, use retrieval-augmented generation.
    return "[route=rag]\n" + answer_rag(question, store, llm)


# Quick confirmation message.
print("Annotated hybrid router ready. Example: ask('What is the highest sell price?', str(csv_path), store, llm)")

## Troubleshooting Appendix

- **Missing modules**: ensure notebook kernel is `.venv` and run `%pip install ...` in that same notebook.
- **`KeyError: UPSTASH_VECTOR_REST_URL`**: run Step 1 again and confirm `.env` contains Upstash keys.
- **Windows path errors (`unicodeescape`)**: use raw strings like `Path(r"C:\\...\\file.csv")`.
- **Long insert times**: use Step 7 batched upserts; reduce `max_source_rows` while testing.
- **Wrong numeric answers from RAG**: use exact analytics path (Step 11 / hybrid analytics route) for max/min/avg/sum queries.